# Gradient Notebook: Architecture B Training on RTX A5000

This notebook trains Architecture B on a Gradient AI notebook with an NVIDIA RTX A5000 GPU.

In [1]:
# Install dependencies (including python-dotenv for .env file support)
!pip install -q torch numpy scipy pandas einops python-dotenv

In [2]:
# Setup environment variables (works in Jupyter/Gradient notebooks)
# This loads WANDB_API_KEY from environment or .env file
import sys
from pathlib import Path
# Add project root to path - UPDATE THIS IF NEEDED
ROOT = Path("/root/attention")  # Change to your project path
sys.path.insert(0, str(ROOT))

from src.utils.env_config import setup_environment
env_status = setup_environment()
print(f"Environment status: {env_status}")

ModuleNotFoundError: No module named 'src'

In [ ]:
import torch

print(f"Project root: {ROOT}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# GPU Configuration
def setup_gpu():
    """Configure GPU for training on RTX A5000."""
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU detected: {gpu_name}")
        print(f"Total VRAM: {gpu_memory:.1f} GB")
        
        # Set optimal CUDA settings for A5000
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.enabled = True
        
        # Use TF32 for better performance on Ampere GPUs
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        
        return torch.device("cuda")
    else:
        print("No GPU detected, falling back to CPU")
        return torch.device("cpu")

device = setup_gpu()
print(f"Using device: {device}")

In [ ]:
# Configuration
CONFIG = {
    "seed": 42,
    "bootstrap_seed": 99,
    "n_episodes": 500,
    "d_embed": 64,
    "hidden_dim": 128,
    "budget_T": 100,
    "write_threshold": 0.5,
    "n_bootstrap": 20000,
    
    "skip_ablations": False,
    "skip_arch_a": False,
    "oracle_ablation": True,
    
    "sanity_check": True,
    "sanity_n_steps": 500,
    "pretrain_episodes": 1000,
    "imitation_episodes": 2000,
    "harden_episodes": 1000,
    "tail_episodes": 1000,
    
    "route_class_weights": [1.0, 10.0, 15.0],
    "phase_lambda": 0.10,
    "write_fp_lambda": 0.05,
    "policy_budget_b": 30,
    
    "eval_free_episodes": 100,
    "litmus": True,
    "litmus_episodes": 200,
    
    "save_checkpoints": True,
}

print("Configuration loaded")

In [ ]:
# Import modules
import datetime
import json
from typing import Dict, List, Optional

from src.utils.seed import set_seed
from src.utils.io import ensure_dir, write_json, write_csv
from src.arch.arch_b import ArchitectureB, FrozenEmbedding, V_TOTAL, V_KEY_VAL
from src.baseline.transformer import ArchitectureA
from src.experiment.distractors import select_hardest_key_distractors
from src.experiment.sweep import sweep_arch_b, sweep_arch_a, aggregate_results, compute_slopes
from src.experiment.ablations import run_ablations
from src.experiment.metrics import CellResult
from src.training.supervised_routing import (
    overfit_one_episode, train_routing_supervised,
    train_routing_imitation, litmus_teacher_forced,
    harden_routing_head, budget_binding_tail,
)

print("All modules imported successfully!")

In [ ]:
# Setup output directory
today = datetime.date.today().strftime("%Y%m%d")
output_dir = Path(f"results/gradient_{today}")
ensure_dir(output_dir)

# Save config
write_json(output_dir / "config.json", CONFIG)

print(f"Output directory: {output_dir}")

In [ ]:
# Set seeds
set_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed(CONFIG["seed"])

# Create embeddings and move to GPU
emb_module = FrozenEmbedding(V_TOTAL, CONFIG["d_embed"], seed=CONFIG["seed"])
embedding_matrix = emb_module.get_matrix().to(device)
print(f"Embedding matrix: {embedding_matrix.shape}")

# Create distractor maps
all_ids = list(range(V_KEY_VAL))
distractor_map_k = select_hardest_key_distractors(
    genuine_key_ids=all_ids,
    embedding_matrix=embedding_matrix.cpu(),
    exclude_ids=set(),
)
distractor_maps = {"K": distractor_map_k, "N": {}}
print(f"Type K distractors: {len(distractor_map_k)} pairs")

In [ ]:
# Model factory
pretrained_state: List[Optional[dict]] = [None]

def arch_b_factory(budget_B: int) -> ArchitectureB:
    model = ArchitectureB(
        d_embed=CONFIG["d_embed"],
        hidden_dim=CONFIG["hidden_dim"],
        budget_B=budget_B,
        budget_T=CONFIG["budget_T"],
        write_threshold=CONFIG["write_threshold"],
        embed_seed=CONFIG["seed"],
    ).to(device)
    
    if pretrained_state[0] is not None:
        model.routing.load_state_dict(pretrained_state[0]["routing"])
        model.output_head.load_state_dict(pretrained_state[0]["output_head"])
        if "gru" in pretrained_state[0]:
            model.gru.load_state_dict(pretrained_state[0]["gru"])
    model.eval()
    model.set_eval_routing(hard=True)
    return model

print("Model factory created")

In [ ]:
# Litmus test
teacher_budget_B = 30
teacher_budget_T = 100

if CONFIG.get("litmus", False):
    n_litmus = CONFIG.get("litmus_episodes", 200)
    print(f"Running litmus test: {n_litmus} episodes")
    
    # Teacher-forced model
    _litmus_model_tf = ArchitectureB(
        d_embed=CONFIG["d_embed"],
        hidden_dim=CONFIG["hidden_dim"],
        budget_B=50,
        budget_T=CONFIG["budget_T"],
        write_threshold=CONFIG["write_threshold"],
        embed_seed=CONFIG["seed"],
    ).to(device)
    
    tf_history = litmus_teacher_forced(
        model=_litmus_model_tf,
        n_episodes=n_litmus,
        lr=3e-3,
        density=0,
        base_seed=CONFIG["seed"],
        ablation_no_mem=False,
        embedding_matrix=embedding_matrix.cpu(),
        log_interval=max(1, n_litmus // 4),
        verbose=True,
    )
    
    # No-memory control model
    _litmus_model_nm = ArchitectureB(
        d_embed=CONFIG["d_embed"],
        hidden_dim=CONFIG["hidden_dim"],
        budget_B=50,
        budget_T=CONFIG["budget_T"],
        write_threshold=CONFIG["write_threshold"],
        embed_seed=CONFIG["seed"],
    ).to(device)
    
    nm_history = litmus_teacher_forced(
        model=_litmus_model_nm,
        n_episodes=n_litmus,
        lr=3e-3,
        density=0,
        base_seed=CONFIG["seed"],
        ablation_no_mem=True,
        embedding_matrix=embedding_matrix.cpu(),
        log_interval=max(1, n_litmus // 4),
        verbose=True,
    )
    
    # Verdict
    w = max(10, n_litmus // 5)
    tf_final = sum(s.accuracy for s in tf_history[-w:]) / w
    nm_final = sum(s.accuracy for s in nm_history[-w:]) / w
    gap = tf_final - nm_final
    passed = tf_final >= 0.20 and gap >= 0.15
    
    print(f"LITMUS: TF={tf_final:.3f}, TF_nomem={nm_final:.3f}, Gap={gap:.3f}")
    print(f"VERDICT: {'PASS' if passed else 'FAIL'}")
    
    write_json(output_dir / "litmus_summary.json", {
        "n_episodes": n_litmus,
        "tf_final_acc": round(tf_final, 4),
        "nm_final_acc": round(nm_final, 4),
        "gap": round(gap, 4),
        "passed": passed,
    })
else:
    print("Skipping litmus test")

In [ ]:
# Sanity check
if CONFIG.get("sanity_check", False):
    print("Running sanity check...")
    sanity_model = arch_b_factory(50)
    n_sanity = CONFIG.get("sanity_n_steps", 500)
    
    sanity_result = overfit_one_episode(
        model=sanity_model,
        seed=CONFIG["seed"],
        n_steps=n_sanity,
        lr=3e-3,
        lambda_route=0.5,
        log_interval=max(1, n_sanity // 10),
        oracle_routing_during_train=True,
        embedding_matrix=embedding_matrix.cpu(),
        verbose=True,
    )
    
    write_json(output_dir / "sanity_check.json", {
        "n_steps": sanity_result.n_steps,
        "converged": sanity_result.converged,
        "final_accuracy": round(sanity_result.final_accuracy, 4),
        "final_skip_frac": round(sanity_result.final_skip_frac, 4),
    })
    print(f"Sanity check: acc={sanity_result.final_accuracy:.3f}")
else:
    print("Skipping sanity check")

In [ ]:
# Pre-training
n_pretrain = CONFIG.get("pretrain_episodes", 0)
if n_pretrain > 0:
    print(f"Pre-training: {n_pretrain} episodes")
    
    pretrain_model = ArchitectureB(
        d_embed=CONFIG["d_embed"],
        hidden_dim=CONFIG["hidden_dim"],
        budget_B=50,
        budget_T=CONFIG["budget_T"],
        write_threshold=CONFIG["write_threshold"],
        embed_seed=CONFIG["seed"],
    ).to(device)
    pretrain_model.train()
    
    pretrain_history = train_routing_supervised(
        model=pretrain_model,
        embedding_matrix=embedding_matrix.cpu(),
        n_episodes=n_pretrain,
        density=0,
        dist_type="N",
        lr=3e-3,
        lambda_route=0.5,
        base_seed=CONFIG["seed"],
        log_interval=max(1, n_pretrain // 10),
        oracle_routing_during_train=True,
        teacher_budget_B=teacher_budget_B,
        teacher_budget_T=teacher_budget_T,
        teacher_align_writes_to_reads=True,
        sparse_targets_only=True,
        policy_budget_b=CONFIG.get("policy_budget_b", 30),
        verbose=True,
    )
    
    pretrained_state[0] = {
        "routing": pretrain_model.routing.state_dict(),
        "output_head": pretrain_model.output_head.state_dict(),
        "gru": pretrain_model.gru.state_dict(),
    }
    
    recent = pretrain_history[-min(50, len(pretrain_history)):]
    avg_acc = sum(s.accuracy for s in recent) / len(recent)
    print(f"Pre-training done. Final avg: acc={avg_acc:.3f}")
else:
    print("Skipping pre-training")

In [ ]:
# Imitation learning
n_imitation = CONFIG.get("imitation_episodes", 0)
if n_imitation > 0:
    print(f"Imitation learning: {n_imitation} episodes")
    
    imitation_model = ArchitectureB(
        d_embed=CONFIG["d_embed"],
        hidden_dim=CONFIG["hidden_dim"],
        budget_B=50,
        budget_T=CONFIG["budget_T"],
        write_threshold=CONFIG["write_threshold"],
        embed_seed=CONFIG["seed"],
    ).to(device)
    
    if pretrained_state[0] is not None:
        imitation_model.routing.load_state_dict(pretrained_state[0]["routing"])
        imitation_model.output_head.load_state_dict(pretrained_state[0]["output_head"])
        imitation_model.gru.load_state_dict(pretrained_state[0]["gru"])
    imitation_model.train()
    
    imitation_history = train_routing_imitation(
        model=imitation_model,
        embedding_matrix=embedding_matrix.cpu(),
        n_episodes=n_imitation,
        density=0,
        dist_type="N",
        lr=1e-3,
        lambda_route=1.5,
        base_seed=CONFIG["seed"] + 10_000,
        log_interval=max(1, n_imitation // 10),
        teacher_budget_B=teacher_budget_B,
        teacher_budget_T=teacher_budget_T,
        teacher_align_writes_to_reads=True,
        sparse_targets_only=True,
        use_oracle_routing=True,
        route_class_weights=CONFIG.get("route_class_weights"),
        lambda_encode_read=CONFIG.get("phase_lambda", 0.10),
        lambda_query_write=CONFIG.get("phase_lambda", 0.10),
        lambda_encode_write_nonoracle=CONFIG.get("write_fp_lambda", 0.05),
        policy_budget_b=CONFIG.get("policy_budget_b", 30),
        verbose=True,
    )
    
    pretrained_state[0] = {
        "routing": imitation_model.routing.state_dict(),
        "output_head": imitation_model.output_head.state_dict(),
        "gru": imitation_model.gru.state_dict(),
    }
    
    recent = imitation_history[-min(50, len(imitation_history)):]
    avg_acc = sum(s.accuracy for s in recent) / len(recent)
    print(f"Imitation done. Final avg: acc={avg_acc:.3f}")
else:
    print("Skipping imitation learning")

In [ ]:
# Routing hardening
n_harden = CONFIG.get("harden_episodes", 0)
if n_harden > 0:
    print(f"Routing hardening: {n_harden} episodes")
    
    harden_model = ArchitectureB(
        d_embed=CONFIG["d_embed"],
        hidden_dim=CONFIG["hidden_dim"],
        budget_B=50,
        budget_T=CONFIG["budget_T"],
        write_threshold=CONFIG["write_threshold"],
        embed_seed=CONFIG["seed"],
    ).to(device)
    
    if pretrained_state[0] is not None:
        harden_model.routing.load_state_dict(pretrained_state[0]["routing"])
        harden_model.output_head.load_state_dict(pretrained_state[0]["output_head"])
        harden_model.gru.load_state_dict(pretrained_state[0]["gru"])
    harden_model.train()
    
    harden_routing_head(
        model=harden_model,
        embedding_matrix=embedding_matrix.cpu(),
        n_episodes=n_harden,
        density=0,
        dist_type="N",
        lr=3e-4,
        lambda_route=3.0,
        base_seed=CONFIG["seed"] + 40_000,
        log_interval=max(1, n_harden // 10),
        teacher_budget_B=teacher_budget_B,
        teacher_budget_T=teacher_budget_T,
        teacher_align_writes_to_reads=True,
        sparse_targets_only=True,
        route_class_weights=CONFIG.get("route_class_weights"),
        policy_budget_b=CONFIG.get("policy_budget_b", 30),
        verbose=True,
    )
    
    pretrained_state[0] = {
        "routing": harden_model.routing.state_dict(),
        "output_head": harden_model.output_head.state_dict(),
        "gru": harden_model.gru.state_dict(),
    }
    
    print("Hardening done.")
else:
    print("Skipping hardening")

In [ ]:
# Budget-binding tail
n_tail = CONFIG.get("tail_episodes", 0)
if n_tail > 0:
    print(f"Budget-binding tail: {n_tail} episodes")
    
    tail_model = ArchitectureB(
        d_embed=CONFIG["d_embed"],
        hidden_dim=CONFIG["hidden_dim"],
        budget_B=50,
        budget_T=CONFIG["budget_T"],
        write_threshold=CONFIG["write_threshold"],
        embed_seed=CONFIG["seed"],
    ).to(device)
    
    if pretrained_state[0] is not None:
        tail_model.routing.load_state_dict(pretrained_state[0]["routing"])
        tail_model.output_head.load_state_dict(pretrained_state[0]["output_head"])
        tail_model.gru.load_state_dict(pretrained_state[0]["gru"])
    tail_model.train()
    
    tail_model, _tail_history = budget_binding_tail(
        model=tail_model,
        embedding_matrix=embedding_matrix.cpu(),
        stage_budgets=(50, 35, 30),
        stage_episodes=(200, 200, 200),
        density=0,
        dist_type="N",
        lr=3e-4,
        lambda_route=1.5,
        base_seed=CONFIG["seed"] + 50_000,
        log_interval=max(1, n_tail // 10),
        teacher_budget_B=teacher_budget_B,
        teacher_budget_T=teacher_budget_T,
        teacher_align_writes_to_reads=True,
        sparse_targets_only=True,
        route_class_weights=CONFIG.get("route_class_weights"),
        lambda_encode_read=CONFIG.get("phase_lambda", 0.10),
        lambda_query_write=CONFIG.get("phase_lambda", 0.10),
        lambda_encode_write_nonoracle=CONFIG.get("write_fp_lambda", 0.05),
        verbose=True,
    )
    
    pretrained_state[0] = {
        "routing": tail_model.routing.state_dict(),
        "output_head": tail_model.output_head.state_dict(),
        "gru": tail_model.gru.state_dict(),
    }
    
    print("Tail phase done.")
else:
    print("Skipping tail phase")

In [ ]:
# Save checkpoint
if pretrained_state[0] is not None and CONFIG.get("save_checkpoints", False):
    checkpoint_path = output_dir / "trained_model.pt"
    torch.save({
        "config": CONFIG,
        "routing": pretrained_state[0]["routing"],
        "output_head": pretrained_state[0]["output_head"],
        "gru": pretrained_state[0]["gru"],
    }, checkpoint_path)
    print(f"Model checkpoint saved to {checkpoint_path}")

In [ ]:
# Architecture B sweep
print(f"Running Architecture B sweep: {CONFIG['n_episodes']} episodes")

ARCH_B_FIELDS = [
    "density", "type", "budget_ratio",
    "accuracy", "ci_low", "ci_high",
    "dcf", "skip_frac", "blocked_attempt_frac",
    "retention", "occupancy",
    "read_frac_q", "write_frac_e",
    "slope", "slope_ci_low", "slope_ci_high",
]

arch_b_full_cells = sweep_arch_b(
    model_factory=arch_b_factory,
    embedding_matrix=embedding_matrix.cpu(),
    distractor_map=distractor_maps,
    n_episodes=CONFIG["n_episodes"],
    base_seed=CONFIG["seed"],
    ablation_flags=None,
)
aggregate_results(arch_b_full_cells, bootstrap_seed=CONFIG["bootstrap_seed"])
compute_slopes(arch_b_full_cells, bootstrap_seed=CONFIG["bootstrap_seed"])

def cell_to_arch_b_row(c: CellResult) -> dict:
    return {
        "density": c.density,
        "type": c.dist_type,
        "budget_ratio": round(c.budget_ratio, 4),
        "accuracy": round(c.accuracy, 5),
        "ci_low": round(c.accuracy_ci_low, 5),
        "ci_high": round(c.accuracy_ci_high, 5),
        "dcf": round(c.dcf, 5),
        "skip_frac": round(c.skip_frac, 5),
        "blocked_attempt_frac": round(c.blocked_attempt_frac, 5),
        "retention": round(c.retention, 5),
        "occupancy": round(c.occupancy, 5),
        "read_frac_q": round(c.read_frac_q, 5),
        "write_frac_e": round(c.write_frac_e, 5),
        "slope": round(c.slope, 5),
        "slope_ci_low": round(c.slope_ci_low, 5),
        "slope_ci_high": round(c.slope_ci_high, 5),
    }

rows_b_full = [cell_to_arch_b_row(c) for c in arch_b_full_cells]
write_csv(output_dir / "arch_b_full.csv", rows_b_full, ARCH_B_FIELDS)
print(f"arch_b_full.csv written ({len(rows_b_full)} rows)")

In [ ]:
# Ablations
if not CONFIG.get("skip_ablations", False):
    print("Running ablations...")
    
    abl_results = run_ablations(
        model_factory=arch_b_factory,
        embedding_matrix=embedding_matrix.cpu(),
        distractor_map_k=distractor_map_k,
        n_episodes=CONFIG["n_episodes"],
        base_seed=CONFIG["seed"],
        bootstrap_seed=CONFIG["bootstrap_seed"],
    )
    
    csv_map = {
        "B_no_mem": "arch_b_no_mem.csv",
        "B_rand": "arch_b_rand.csv",
        "B_sched": "arch_b_sched.csv",
        "B_zero_h": "arch_b_zero_h.csv",
        "B_no_sal": "arch_b_no_sal.csv",
    }
    if CONFIG.get("oracle_ablation", False):
        csv_map["B_oracle"] = "arch_b_oracle.csv"
        csv_map["B_force_read"] = "arch_b_force_read.csv"
    
    for abl_name, fname in csv_map.items():
        cells = abl_results.get(abl_name, [])
        rows = [cell_to_arch_b_row(c) for c in cells]
        write_csv(output_dir / fname, rows, ARCH_B_FIELDS)
        print(f"{fname} written ({len(rows)} rows)")
else:
    print("Skipping ablations")

In [ ]:
# Architecture A
if not CONFIG.get("skip_arch_a", False):
    print("Running Architecture A sweep...")
    
    arch_a = ArchitectureA(d_model=64, n_layers=12, n_heads=8, embed_seed=CONFIG["seed"])
    arch_a = arch_a.to(device)
    arch_a.eval()
    
    ARCH_A_FIELDS = [
        "density", "type",
        "accuracy", "ci_low", "ci_high",
        "skip_frac",
        "slope", "slope_ci_low", "slope_ci_high",
    ]
    
    arch_a_results = sweep_arch_a(
        model=arch_a,
        embedding_matrix=embedding_matrix.cpu(),
        distractor_map=distractor_maps,
        n_episodes=CONFIG["n_episodes"],
        base_seed=CONFIG["seed"],
    )
    aggregate_results(arch_a_results, bootstrap_seed=CONFIG["bootstrap_seed"])
    compute_slopes(arch_a_results, bootstrap_seed=CONFIG["bootstrap_seed"])
    
    def cell_to_arch_a_row(c: CellResult) -> dict:
        return {
            "density": c.density,
            "type": c.dist_type,
            "accuracy": round(c.accuracy, 5),
            "ci_low": round(c.accuracy_ci_low, 5),
            "ci_high": round(c.accuracy_ci_high, 5),
            "skip_frac": round(c.skip_frac, 5),
            "slope": round(c.slope, 5),
            "slope_ci_low": round(c.slope_ci_low, 5),
            "slope_ci_high": round(c.slope_ci_high, 5),
        }
    
    rows_a = [cell_to_arch_a_row(c) for c in arch_a_results]
    write_csv(output_dir / "arch_a.csv", rows_a, ARCH_A_FIELDS)
    print(f"arch_a.csv written ({len(rows_a)} rows)")
else:
    print("Skipping Architecture A")

In [ ]:
# Summary
print(f"\nTraining Complete!")
print(f"Results saved to: {output_dir}")

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

In [ ]:
# View results
import pandas as pd
import os

print("Output files:")
for f in sorted(os.listdir(output_dir)):
    fpath = output_dir / f
    size = os.path.getsize(fpath) / 1024
    print(f"  {f}: {size:.1f} KB")

In [ ]:
# Display Architecture B results
arch_b_df = pd.read_csv(output_dir / "arch_b_full.csv")
print("Architecture B Results:")
print(arch_b_df.head(20))